# MPC minutes preprocessing
Load, clean, and tokenize the MPC minutes according to the assignment instructions.

In [ ]:
# Imports and NLTK resources
from pathlib import Path
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Ensure required tokenizers and corpora are available
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [3]:
# Load raw MPC minutes
data_path = Path("../raw/mpc_minutes.txt")
raw_text = data_path.read_text(encoding="utf-8", errors="ignore")
documents = [line.strip() for line in raw_text.splitlines() if line.strip()]
print(f"Loaded {len(documents)} documents from {data_path}")

Loaded 7278 documents from ../raw/mpc_minutes.txt


In [4]:
# Contraction expansion helper
try:
    import contractions
    def expand_contractions(text: str) -> str:
        return contractions.fix(text)
except ImportError:
    CONTRACTION_MAP = {
        "n't": " not",
        "'re": " are",
        "'s": " is",
        "'d": " would",
        "'ll": " will",
        "'ve": " have",
        "'m": " am"
    }
    contraction_pattern = re.compile("|".join(map(re.escape, CONTRACTION_MAP.keys())))
    def expand_contractions(text: str) -> str:
        return contraction_pattern.sub(lambda m: CONTRACTION_MAP[m.group(0)], text)

In [5]:
# Tokenization and normalization pipeline
def preprocess_document(text: str) -> dict:
    text = text.lower()
    text = expand_contractions(text)
    tokens = word_tokenize(text)
    cleaned = []
    for tok in tokens:
        tok_ascii = tok.encode("ascii", "ignore").decode("ascii")
        if not tok_ascii:
            continue
        if len(tok_ascii) < 3:
            continue
        if not tok_ascii.isalpha():
            continue
        if tok_ascii in stop_words:
            continue
        cleaned.append(tok_ascii)
    stemmed = [stemmer.stem(t) for t in cleaned]
    lemmatized = [lemmatizer.lemmatize(t) for t in cleaned]
    return {
        "tokens": cleaned,
        "stemmed": stemmed,
        "lemmatized": lemmatized
    }

In [6]:
# Apply preprocessing and inspect a sample
processed_docs = [preprocess_document(doc) for doc in documents]
print(f"Preprocessed {len(processed_docs)} documents")

if processed_docs:
    sample = processed_docs[0]
    print("Tokens (first doc, first 30):", sample["tokens"][:30])
    print("Stemmed (first 30):", sample["stemmed"][:30])
    print("Lemmas (first 30):", sample["lemmatized"][:30])

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/Users/kaibiaozhu/nltk_data'
    - '/opt/anaconda3/envs/5020_env/nltk_data'
    - '/opt/anaconda3/envs/5020_env/share/nltk_data'
    - '/opt/anaconda3/envs/5020_env/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
